# 🗂️ **Skripsi** : ANALISIS SENTIMEN — RANDOM FOREST CLASSIFIER (OPTIMIZED STANDALONE)

**Author**   : [Nofendra Tahta Dirgantara]

**Pipeline** : Load → Clean → Preprocess → Label → EDA → Model → Tune → Save → Infer

--- 

### 💡 **Deskripsi Notebook**
Notebook ini adalah versi teroptimasi dari pipeline analisis sentimen beasiswa LPDP. Semua logic optimasi (Sastrawi dictionary patch, custom lexicon overrides, class weight balancing, hyperparameter tuning, dan probability blending inference) telah terintegrasi langsung di dalam sel notebook sehingga notebook ini bersifat **standalone** dan dapat dijalankan langsung (misalnya di Google Colab atau Jupyter) tanpa membutuhkan berkas eksternal dari folder `src`.

In [ ]:
# ── INSTALL DEPENDENCY (Untuk Google Colab jika diperlukan) ────
# !pip install sastrawi wordcloud langdetect joblib scikit-learn seaborn matplotlib -q

# ⚙️ IMPORT LIBRARY & GLOBAL CONFIGURATION

In [ ]:
import re
import os
import json
import warnings
import joblib
import requests
from io import StringIO
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from langdetect import detect, LangDetectException

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Konfigurasi Global
warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Warna konsisten untuk 3 kelas sentimen
SENTIMENT_COLORS = {
    "Positif": "#2ecc71",
    "Netral":  "#3498db",
    "Negatif": "#e74c3c",
}

print("✅ Semua library berhasil diimport.")

# 📥 LOAD DATASET

In [ ]:
DATASET_URL = (
    "https://raw.githubusercontent.com/go0se05/Analysis-Sentiment_LPDP"
    "/refs/heads/main/master_data_lpdp.csv"
)
COLS_TO_USE = ["id_str", "created_at", "full_text", "lang"]

print("📥 Memuat dataset dari GitHub ...")
df = pd.read_csv(DATASET_URL, low_memory=False)

# Pastikan kolom yang dibutuhkan tersedia dan ambil yang relevan
df = df[COLS_TO_USE].copy()

print(f"\n📊 Informasi Dataset Awal:")
print(f"  Jumlah baris   : {df.shape[0]:,}")
print(f"  Jumlah kolom   : {df.shape[1]}")
print(f"  Kolom          : {list(df.columns)}")
print(f"\n🔍 5 Data Pertama:")
df.head(5)


# 🧼 DATA CLEANING

### Cek Missing Values & Duplikat

In [ ]:
print("=" * 55)
print("📋 STATUS AWAL DATA")
print("=" * 55)
print(f"  Total baris            : {len(df):,}")
print(f"  Duplikat (full_text)   : {df.duplicated(subset='full_text').sum():,}")
print(f"\n  Missing values per kolom:")
print(df.isnull().sum().to_string())


### Hapus NA & Duplikat

In [ ]:
before = len(df)

# Langkah 1 — Hapus baris dengan missing value di kolom full_text
df = df.dropna(subset=["full_text"]).copy()
after_na = len(df)

# Langkah 2 — Hapus duplikat berdasarkan teks
df = df.drop_duplicates(subset="full_text", keep="first").reset_index(drop=True)
after_dup = len(df)

print("=" * 55)
print("✅ STATUS SETELAH CLEANING")
print("=" * 55)
print(f"  Baris awal             : {before:,}")
print(f"  Setelah hapus NA       : {after_na:,}  (dihapus: {before - after_na:,})")
print(f"  Setelah hapus duplikat : {after_dup:,}  (dihapus: {after_na - after_dup:,})")
print(f"  Total baris bersih     : {after_dup:,}")
print(f"\n  Missing values tersisa:")
print(df.isnull().sum().to_string())

print(f"\n📄 Contoh data bersih:")
df.head(3)


# ⚙️ PREPROCESSING (NLP INDONESIA)

### Performance Monkey-Patch untuk Sastrawi ArrayDictionary
Stemmer Sastrawi secara default berjalan sangat lambat karena pencarian kamus menggunakan list linear. Kita me-monkey-patch Sastrawi agar menggunakan `set` (hashing) sehingga lookup kamus menjadi $O(1)$ dan stemming bisa berjalan 100x lebih cepat.

In [ ]:
from Sastrawi.Dictionary.ArrayDictionary import ArrayDictionary

def _patched_init(self, words=None):
    self.words = set()
    if words:
        self.add_words(words)

def _patched_contains(self, word):
    if not word or word.strip() == '':
        return False
    return word in self.words

def _patched_count(self):
    return len(self.words)

def _patched_add_words(self, words):
    clean_words = {w for w in words if w and w.strip() != ''}
    self.words.update(clean_words)

def _patched_add(self, word):
    if not word or word.strip() == '':
        return
    self.words.add(word)

ArrayDictionary.__init__ = _patched_init
ArrayDictionary.contains = _patched_contains
ArrayDictionary.count = _patched_count
ArrayDictionary.add_words = _patched_add_words
ArrayDictionary.add = _patched_add

print("⚡ Sastrawi ArrayDictionary berhasil dipatch dengan Set (pencarian O(1))!")

### Definisi Kelas `TextPreprocessor` Teroptimasi
Kelas ini menangani pre-processing data tweet secara lengkap.

In [ ]:
# Kata-kata bahasa Indonesia yang dominan untuk deteksi bahasa secara cepat (heuristic)
STRONG_ID_WORDS = {
    "yg", "dgn", "nya", "jg", "juga", "ini", "itu", "dan", "atau", "untuk",
    "dengan", "ada", "tidak", "bisa", "aja", "gak", "ga", "gw", "aku",
    "kamu", "saya", "kalian", "kita", "mereka", "dia", "udah", "sudah",
    "jadi", "sih", "deh", "nih", "lah", "dong", "banget", "bgt"
}

# Kamus normalisasi kata alay/singkatan offline (fallback)
DEFAULT_KAMUS_BAKU = {
    "gak": "tidak", "ga": "tidak", "udah": "sudah",
    "gimana": "bagaimana", "kalo": "kalau", "aja": "saja",
    "bgt": "banget", "yg": "yang", "dgn": "dengan",
    "tdk": "tidak", "blm": "belum", "sdh": "sudah",
    "krn": "karena", "utk": "untuk", "lg": "lagi",
    "emg": "memang", "ttp": "tetap", "hrs": "harus",
    "pake": "pakai", "pengen": "ingin", "kmrn": "kemarin",
    "abis": "habis", "bikin": "membuat", "nggak": "tidak",
}

# Stopwords kustom khusus konteks Twitter/Sosmed Indonesia
CUSTOM_STOPWORDS = {
    "yg", "dgn", "nya", "jg", "juga", "ini", "itu", "dan", "atau", "untuk",
    "dengan", "ada", "tidak", "bisa", "aja", "gak", "ga", "gw", "aku",
    "kamu", "saya", "kalian", "kita", "mereka", "dia", "udah", "sudah",
    "jadi", "ya", "sih", "deh", "nih", "lah", "dong", "banget", "bgt",
    "wkwk", "haha", "lol", "wah", "oh", "ah", "eh", "hm",
    "rt", "amp", "via", "cc", "https", "http", "co", "www",
}

class TextPreprocessor:
    def __init__(self, kamus_url="https://raw.githubusercontent.com/go0se05/PI/main/kamuskatabaku.csv"):
        # Inisialisasi stemmer Sastrawi
        stemmer_factory = StemmerFactory()
        self.stemmer = stemmer_factory.create_stemmer()
        
        # Inisialisasi Stopwords
        sw_factory = StopWordRemoverFactory()
        self.stopwords_set = set(sw_factory.get_stop_words())
        self.stopwords_set.update(CUSTOM_STOPWORDS)
        
        # Muat Kamus Normalisasi
        self.kamus_baku = self._load_normalization_dict(kamus_url)
        
        # Precompile pola regex untuk optimasi performa
        self.url_pattern = re.compile(r"http\S+|www\S+")
        self.mention_pattern = re.compile(r"@\w+")
        self.hashtag_pattern = re.compile(r"#(\w+)")
        self.emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"
            "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF"
            "\U0001F700-\U0001F77F"
            "\U0001F780-\U0001F7FF"
            "\U0001F800-\U0001F8FF"
            "\U0001F900-\U0001F9FF"
            "\U0001FA00-\U0001FA6F"
            "\U0001FA70-\U0001FAFF"
            "\U00002702-\U000027B0"
            "]+",
            flags=re.UNICODE,
        )
        self.digits_pattern = re.compile(r"\d+")
        self.non_alpha_pattern = re.compile(r"[^\w\s]")
        self.underscores_pattern = re.compile(r"_+")
        self.spaces_pattern = re.compile(r"\s+")

    def _load_normalization_dict(self, url):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            df_k = pd.read_csv(StringIO(r.text))
            cols = df_k.columns.str.lower().tolist()
            if "tidak_baku" in cols and "kata_baku" in cols:
                mapping = dict(zip(df_k["tidak_baku"], df_k["kata_baku"]))
            else:
                mapping = dict(zip(df_k.iloc[:, 0], df_k.iloc[:, 1]))
            return mapping
        except Exception as e:
            print(f"⚠️ Gagal memuat kamus kata baku dari URL ({e}). Menggunakan kamus fallback.")
            return DEFAULT_KAMUS_BAKU

    def filter_text(self, text):
        if not isinstance(text, str):
            return ""
        text = self.url_pattern.sub(" ", text)
        text = self.mention_pattern.sub(" ", text)
        text = self.hashtag_pattern.sub(r"\1", text)
        text = self.emoji_pattern.sub(" ", text)
        text = self.digits_pattern.sub(" ", text)
        text = self.non_alpha_pattern.sub(" ", text)
        text = self.underscores_pattern.sub(" ", text)
        text = self.spaces_pattern.sub(" ", text).strip()
        return text

    def is_indonesian(self, text):
        words = text.split()
        if len(words) < 3:
            return True
        # Deteksi cepat lewat kecocokan kata bahasa Indonesia umum
        id_words_count = sum(1 for w in words if w in STRONG_ID_WORDS)
        if id_words_count >= 2:
            return True
        try:
            return detect(text) == "id"
        except LangDetectException:
            return True

    def normalize_words(self, text):
        words = text.split()
        normalized = [self.kamus_baku.get(w, w) for w in words]
        return " ".join(normalized)

    def remove_stopwords(self, text):
        words = text.split()
        filtered = [w for w in words if w not in self.stopwords_set and len(w) > 1]
        return " ".join(filtered)

    def stem(self, text):
        if not text.strip():
            return ""
        return self.stemmer.stem(text)

    def preprocess(self, text, filter_lang=True):
        text_clean = self.filter_text(text)
        if not text_clean:
            return ""
        text_lower = text_clean.lower()
        if filter_lang and not self.is_indonesian(text_lower):
            return ""
        text_norm = self.normalize_words(text_lower)
        text_sw = self.remove_stopwords(text_norm)
        text_stem = self.stem(text_sw)
        return text_stem


In [ ]:
print("⚙️ Menginisialisasi TextPreprocessor ...")
preprocessor = TextPreprocessor()

print("\n🚀 Menjalankan pipeline preprocessing lengkap pada kolom full_text...")
df["processed_text"] = df["full_text"].apply(preprocessor.preprocess)

# Menghapus teks yang menjadi kosong setelah preprocessing
before_empty_filter = len(df)
df = df[df["processed_text"].str.strip().str.len() > 0].reset_index(drop=True)
after_empty_filter = len(df)

print(f"\n✅ Preprocessing selesai!")
print(f"  Baris sebelum filter kosong: {before_empty_filter:,}")
print(f"  Baris setelah filter kosong: {after_empty_filter:,} (Dihapus: {before_empty_filter - after_empty_filter:,})")
print(f"\n🔍 Contoh hasil preprocessing:")
df[["full_text", "processed_text"]].head(5)


# 🏷️ PELABELAN (LEXICON-BASED SENTIMENT LABELING)

### Definisi Kelas `LexiconLabeler` dengan Overrides Kustom
Kelas ini memuat kamus sentimen positif & negatif dari **InSet (Fajri et al.)**, memecahkan konflik kata bertumpukan, dan menerapkan override kustom agar istilah domain-spesifik (*lpdp*, *beasiswa*, *seleksi*, *awardee*, dll.) bernilai netral (0) untuk mereduksi bias klasifikasi.

In [ ]:
DEFAULT_POS_LEXICON = {
    "bagus": 3, "senang": 4, "hebat": 4, "cinta": 5, "setuju": 3, "baik": 3,
    "lolos": 4, "sukses": 4, "semangat": 4, "terima kasih": 5, "membantu": 3,
    "bangga": 4, "alhamdulillah": 5, "untung": 3, "mudah": 3, "menang": 4,
}

DEFAULT_NEG_LEXICON = {
    "jelek": -3, "marah": -4, "benci": -5, "kecewa": -4, "gagal": -4,
    "susah": -3, "ribet": -3, "buruk": -3, "rugi": -3, "lemah": -2,
    "mahal": -2, "salah": -2, "mengecewakan": -5, "amburadul": -4,
}

class LexiconLabeler:
    def __init__(
        self,
        pos_url="https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv",
        neg_url="https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"
    ):
        self.lexicon_pos = self._load_tsv_lexicon(pos_url, is_positive=True)
        self.lexicon_neg = self._load_tsv_lexicon(neg_url, is_positive=False)
        self._resolve_conflicts()
        self._apply_overrides()

    def _load_tsv_lexicon(self, url, is_positive):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            df_l = pd.read_csv(StringIO(r.text), sep="\t", header=None, names=["word", "weight"])
            df_l["word"] = df_l["word"].astype(str).str.lower().str.strip()
            df_l["weight"] = pd.to_numeric(df_l["weight"], errors="coerce")
            df_l = df_l.dropna(subset=["weight"])
            df_l["weight"] = df_l["weight"].astype(int)
            return dict(zip(df_l["word"], df_l["weight"]))
        except Exception as e:
            print(f"⚠️ Gagal memuat leksikon dari URL ({e}). Menggunakan fallback lokal.")
            return DEFAULT_POS_LEXICON if is_positive else DEFAULT_NEG_LEXICON

    def _resolve_conflicts(self):
        conflicts = set(self.lexicon_pos.keys()) & set(self.lexicon_neg.keys())
        for word in conflicts:
            pos_val = self.lexicon_pos[word]
            neg_val = abs(self.lexicon_neg[word])
            if pos_val > neg_val:
                self.lexicon_pos[word] = pos_val - neg_val
                self.lexicon_neg.pop(word, None)
            elif neg_val > pos_val:
                self.lexicon_neg[word] = -(neg_val - pos_val)
                self.lexicon_pos.pop(word, None)
            else:
                self.lexicon_pos.pop(word, None)
                self.lexicon_neg.pop(word, None)

    def _apply_overrides(self):
        # Kustomisasi bobot leksikon untuk mengurangi bias sentimen dan noise
        lexicon_overrides = {
            "lolos": 4, "bantu": 4, "membantu": 4, "benar": 2, "bangga": 4, "alhamdulillah": 5,
            "bagus": 4, "sukses": 4, "hebat": 4, "cinta": 5, "setuju": 3, "baik": 3,
            "semangat": 4, "terimakasih": 5, "untung": 3, "mudah": 3, "menang": 4,
            "bahagia": 4, "cocok": 3, "kuat": 3, "luar biasa": 4, "senang": 4,
            "gembira": 4, "puji": 3, "bersyukur": 5, "aman": 3, "damai": 3, "maju": 3,
            "pintar": 3, "cerdas": 4, "kreatif": 3,
            "kecewa": -4, "mengecewakan": -5, "ribet": -3, "buruk": -4, "jelek": -3,
            "gagal": -4, "susah": -3, "sulit": -3, "rugi": -3, "lemah": -2, "mahal": -2,
            "salah": -2, "benci": -5, "marah": -4, "amburadul": -4, "parah": -3,
            "payah": -3, "hancur": -4, "kacau": -4, "masalah": -3, "permasalahan": -3,
            "tuntut": -2, "tuntutan": -2, "sesal": -3, "menyesal": -3,
            "cukup": 0, "anak": 0, "jangan": 0, "tidak": 0, "gak": 0, "ga": 0, "nggak": 0,
            "apa": 0, "semua": 0, "ikut": 0, "kalau": 0, "kalo": 0, "kayak": 0, "pilih": 0,
            "kata": 0, "masuk": 0, "pajak": 0, "sama": 0, "punya": 0, "buat": 0, "bikin": 0,
            "laku": 0, "lakukan": 0, "ada": 0, "adalah": 0, "yaitu": 0, "merupakan": 0,
            "menjadi": 0, "jadi": 0, "bisa": 0, "dapat": 0, "luar": 0, "negeri": 0,
            "dalam": 0, "negara": 0, "indonesia": 0, "warga": 0, "rakyat": 0, "lpdp": 0,
            "beasiswa": 0, "awardee": 0, "penerima": 0, "penerimabeasiswa": 0, "pendaftaran": 0,
            "daftar": 0, "seleksi": 0, "syarat": 0, "reguler": 0, "kuota": 0, "kuliah": 0,
            "sekolah": 0, "studi": 0, "belajar": 0, "pendidikan": 0, "didik": 0, "universitas": 0,
            "kampus": 0, "uras": 0, "brain": 0, "drain": 0, "alumni": 0, "lulusan": 0,
            "video": 0, "unggah": 0, "posting": 0, "cuitan": 0, "tweet": 0, "netizen": 0,
            "warganet": 0, "bapak": 0, "ibu": 0, "orang": 0, "orangtua": 0, "keluarga": 0,
            "tahun": 0, "bulan": 0, "hari": 0, "jam": 0, "tanggal": 0, "juni": 0, "juli": 0,
            "waktu": 0, "kali": 0, "dulu": 0, "sekarang": 0, "nanti": 0, "besok": 0,
            "kemarin": 0, "lusa": 0, "info": 0, "bilang": 0, "lanjut": 0, "akhir": 0, "batas": 0,
            "sedia": 0, "lengkap": 0, "resmi": 0, "buka": 0, "banyak": 0, "proses": 0, "sangat": 0,
            "sering": 0, "pemberitahuan": 0, "jelas": 0, "dokumen": 0, "admin": 0, "adminnya": 0,
            "ubah": 0, "berubah": 0, "pelamar": 0, "lamar": 0, "birokrasi": 0
        }
        for word, weight in lexicon_overrides.items():
            if weight > 0:
                self.lexicon_pos[word] = weight
                self.lexicon_neg.pop(word, None)
            elif weight < 0:
                self.lexicon_neg[word] = weight
                self.lexicon_pos.pop(word, None)
            else:
                self.lexicon_pos.pop(word, None)
                self.lexicon_neg.pop(word, None)

    def label_sentiment(self, text):
        if not isinstance(text, str) or not text.strip():
            return 0, "Netral"
        score = 0
        words = text.split()
        for token in words:
            score += self.lexicon_pos.get(token, 0)
            score -= abs(self.lexicon_neg.get(token, 0))
        if score > 0:
            return score, "Positif"
        elif score < 0:
            return score, "Negatif"
        else:
            return 0, "Netral"


In [ ]:
print("🏷️ Menginisialisasi LexiconLabeler ...")
labeler = LexiconLabeler()

print("\n⏳ Menghitung skor sentimen dan menetapkan label kelas ...")
lexicon_results = [labeler.label_sentiment(t) for t in df["processed_text"]]
df["Score"] = [r[0] for r in lexicon_results]
df["label"] = [r[1] for r in lexicon_results]

print("\n✅ Pelabelan selesai!")
dist = df["label"].value_counts()
print("📊 Distribusi Kelas Sentimen Hasil Pelabelan Leksikon:")
for kls, jml in dist.items():
    pct = (jml / len(df)) * 100
    print(f"  - {kls:<8}: {jml:>6,} ({pct:>5.1f}%)")


# 📊 EXPLORATORY DATA ANALYSIS (EDA)

### 1. Distribusi Kelas Sentimen (Bar & Pie Chart)

In [ ]:
dist = df["label"].value_counts()
labels = dist.index.tolist()
counts = dist.values.tolist()
colors = [SENTIMENT_COLORS.get(l, "#95a5a6") for l in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribusi Kelas Sentimen Tweet LPDP", fontsize=14, fontweight="bold")

# Bar chart
bars = axes[0].bar(labels, counts, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_xlabel("Kelas Sentimen", fontsize=11)
axes[0].set_ylabel("Jumlah Tweet", fontsize=11)
axes[0].set_title("Bar Chart", fontsize=12)
for bar, count in zip(bars, counts):
    percentage = (count / len(df)) * 100
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{count:,}\n({percentage:.1f}%)",
        ha="center", va="bottom", fontsize=10,
    )

# Pie chart
axes[1].pie(
    counts, labels=labels, colors=colors, autopct="%1.1f%%",
    startangle=140, wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    textprops={"fontsize": 11},
)
axes[1].set_title("Pie Chart", fontsize=12)

plt.tight_layout()
plt.show()


### 2. WordCloud Keseluruhan & Per Kelas Sentimen

In [ ]:
# WordCloud Semua Tweet
corpus_all = " ".join(df["processed_text"].dropna())
if corpus_all.strip():
    wc_all = WordCloud(
        background_color="white",
        max_words=200,
        colormap="viridis",
        width=1000, height=500,
        collocations=False,
    ).generate(corpus_all)
    
    plt.figure(figsize=(12, 6))
    plt.imshow(wc_all, interpolation="bilinear")
    plt.axis("off")
    plt.title("WordCloud — Semua Tweet LPDP (setelah preprocessing)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

# WordCloud Per Kelas Sentimen
cmap_map = {"Positif": "Greens", "Netral": "Blues", "Negatif": "Reds"}
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("WordCloud per Kelas Sentimen", fontsize=14, fontweight="bold")

classes = ["Positif", "Netral", "Negatif"]
for i, kelas in enumerate(classes):
    ax = axes[i]
    subset = df[df["label"] == kelas]["processed_text"]
    if subset.empty or not " ".join(subset.dropna()).strip():
        ax.axis("off")
        ax.set_title(f"{kelas} (tidak ada data)", fontsize=11)
        continue
    corpus = " ".join(subset.dropna())
    wc = WordCloud(
        background_color="white", max_words=150,
        colormap=cmap_map.get(kelas, "Blues"),
        width=800, height=400, collocations=False,
    ).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"Sentimen: {kelas} ({len(subset):,} tweet)", fontsize=11, fontweight="bold", color=SENTIMENT_COLORS.get(kelas, "black"))

plt.tight_layout()
plt.show()


### 3. Frekuensi Kata Terbanyak (Top-20 Bar Chart)

In [ ]:
all_words = " ".join(df["processed_text"].dropna()).split()
if all_words:
    freq = Counter(all_words).most_common(20)
    words_top, counts_top = zip(*freq)
    
    plt.figure(figsize=(13, 7))
    palette = sns.color_palette("mako_r", len(words_top))
    bars = plt.barh(list(words_top)[::-1], list(counts_top)[::-1], color=palette)
    plt.xlabel("Frekuensi Kemunculan", fontsize=11)
    plt.title("Top-20 Kata Paling Sering Muncul (setelah preprocessing)", fontsize=13, fontweight="bold")
    
    for bar, count in zip(bars, list(counts_top)[::-1]):
        plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f"{count:,}", va="center", fontsize=9)
    
    plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.tight_layout()
    plt.show()


### 4. Tren Sentimen per Waktu

In [ ]:
if "created_at" in df.columns:
    df_time = df.copy()
    df_time["created_at"] = pd.to_datetime(df_time["created_at"], errors="coerce", utc=True)
    df_time = df_time.dropna(subset=["created_at"])
    
    if not df_time.empty:
        # Tren per Bulan
        df_time["bulan"] = df_time["created_at"].dt.to_period("M").astype(str)
        pivot_bulan = df_time.groupby(["bulan", "label"]).size().unstack(fill_value=0).sort_index()
        
        plt.figure(figsize=(15, 5))
        for kelas in ["Positif", "Netral", "Negatif"]:
            if kelas in pivot_bulan.columns:
                plt.plot(pivot_bulan.index.astype(str), pivot_bulan[kelas], marker="o", markersize=5, linewidth=2, label=kelas, color=SENTIMENT_COLORS[kelas])
        plt.title("Tren Sentimen Tweet LPDP per Bulan", fontsize=13, fontweight="bold")
        plt.xlabel("Bulan")
        plt.ylabel("Jumlah Tweet")
        plt.legend(title="Sentimen")
        plt.xticks(rotation=45)
        plt.grid(axis="y", linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()
        
        # Tren per Hari & Jam
        df_time["hari"] = df_time["created_at"].dt.day_name()
        df_time["jam"]  = df_time["created_at"].dt.hour
        hari_urut = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle("Tren Sentimen Berdasarkan Hari & Jam", fontsize=13, fontweight="bold")
        
        pivot_hari = df_time.groupby(["hari", "label"]).size().unstack(fill_value=0).reindex([h for h in hari_urut if h in df_time["hari"].unique()])
        for kelas in ["Positif", "Netral", "Negatif"]:
            if kelas in pivot_hari.columns:
                axes[0].plot(pivot_hari.index.astype(str), pivot_hari[kelas], marker="o", markersize=5, linewidth=2, label=kelas, color=SENTIMENT_COLORS[kelas])
        axes[0].set_title("Per Hari dalam Seminggu")
        axes[0].set_xlabel("Hari")
        axes[0].set_ylabel("Jumlah Tweet")
        axes[0].legend(title="Sentimen")
        axes[0].tick_params(axis="x", rotation=30)
        axes[0].grid(axis="y", linestyle="--", alpha=0.5)
        
        pivot_jam = df_time.groupby(["jam", "label"]).size().unstack(fill_value=0).sort_index()
        for kelas in ["Positif", "Netral", "Negatif"]:
            if kelas in pivot_jam.columns:
                axes[1].plot(pivot_jam.index.astype(str), pivot_jam[kelas], marker="o", markersize=5, linewidth=2, label=kelas, color=SENTIMENT_COLORS[kelas])
        axes[1].set_title("Per Jam dalam Sehari")
        axes[1].set_xlabel("Jam")
        axes[1].set_ylabel("Jumlah Tweet")
        axes[1].legend(title="Sentimen")
        axes[1].tick_params(axis="x", rotation=45)
        axes[1].grid(axis="y", linestyle="--", alpha=0.5)
        
        plt.tight_layout()
        plt.show()


# 🔧 DATA PREPARATION (TF-IDF & ENCODING)

In [ ]:
print("🔧 Menyiapkan data untuk pemodelan ...")

# Mengisi NaN di kolom teks dengan string kosong
df["processed_text"] = df["processed_text"].fillna("")
X = df["processed_text"]
y_raw = df["label"]

# ── 1. Label Encoding ────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(y_raw)
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"  Label Mapping  : {label_mapping}")

# ── 2. TF-IDF Vectorization ──────────────────────────────────────
print("\n Fitting TF-IDF Vectorizer (max_features=20000, ngram_range=(1,2))...")
tfidf = TfidfVectorizer(
    sublinear_tf=True,
    min_df=3,
    max_features=20000,
    ngram_range=(1, 2),
    analyzer="word",
)
X_tfidf = tfidf.fit_transform(X)
print(f"  Shape matriks TF-IDF: {X_tfidf.shape}")

# ── 3. Train-Test Split (80/20 Stratified) ─────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"  Data Train: {X_train.shape[0]} | Data Test: {X_test.shape[0]}")


# 🤖 BUILD & TRAIN

### Base Random Forest Model
Melatih model awal Random Forest Classifier dengan `class_weight='balanced'` untuk menangani class imbalance secara otomatis.

In [ ]:
print("🤖 Melatih model Random Forest (Base) ...")

rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_base.fit(X_train, y_train)

print("\n✅ Training Base selesai!")
print(f"  Jumlah kelas   : {rf_base.n_classes_}")
print(f"  Kelas          : {list(le.classes_)}")


# 📊 EVALUASI MODEL DASAR

In [ ]:
y_pred_base = rf_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)

print(f"Base Model Accuracy: {acc_base:.4%}")
print("\n📊 Classification Report (Base Model):")
print(classification_report(y_test, y_pred_base, target_names=le.classes_))

# Plot Confusion Matrix
cm_base = confusion_matrix(y_test, y_pred_base)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_base, annot=True, fmt="d", cmap="Blues",
    xticklabels=le.classes_, yticklabels=le.classes_,
    linewidths=0.5, linecolor="white",
    annot_kws={"size": 12, "weight": "bold"},
)
plt.xlabel("Prediksi", fontsize=11)
plt.ylabel("Aktual", fontsize=11)
plt.title(f"Confusion Matrix — Base Model (Akurasi: {acc_base*100:.2f}%)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


# ⚙️ HYPERPARAMETER TUNING

### Setup & Eksekusi `RandomizedSearchCV`
Untuk mengoptimasi performa model, kita melangsungkan Randomized Search sebanyak 30 iterasi dengan CV=5. Metrik evaluasi yang digunakan adalah **F1 Macro** yang sangat sensitif terhadap ketidakseimbangan kelas. Model dituning menggunakan `class_weight='balanced_subsample'`.

In [ ]:
print("⏳ Menjalankan RandomizedSearchCV hyperparameter tuning (30 iterasi, 5-Fold CV)...")

param_distributions = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 10, 20, 30, 50],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 6],
    "max_features": ["sqrt", "log2", 0.3, 0.5],
    "bootstrap": [True, False],
}

rf_tuning = RandomForestClassifier(
    class_weight="balanced_subsample",
    n_jobs=None,
    random_state=RANDOM_STATE,
)

search = RandomizedSearchCV(
    estimator=rf_tuning,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="f1_macro",
    cv=5,
    refit=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)
search.fit(X_train, y_train)

rf_tuned = search.best_estimator_
best_params = search.best_params_
print(f"\n✅ Tuning selesai! Score F1-macro Terbaik CV: {search.best_score_:.4f}")
print(f"   Best Parameters: {best_params}")


# 📊 EVALUASI MODEL TERBAIK (TUNED)

In [ ]:
y_pred_tuned = rf_tuned.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"Tuned Model Accuracy: {acc_tuned:.4%}")
print("\n📊 Classification Report (Tuned Model):")
print(classification_report(y_test, y_pred_tuned, target_names=le.classes_))

# Plot Confusion Matrix Tuned
cm_tuned = confusion_matrix(y_test, y_pred_tuned)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_tuned, annot=True, fmt="d", cmap="Greens",
    xticklabels=le.classes_, yticklabels=le.classes_,
    linewidths=0.5, linecolor="white",
    annot_kws={"size": 12, "weight": "bold"},
)
plt.xlabel("Prediksi", fontsize=11)
plt.ylabel("Aktual", fontsize=11)
plt.title(f"Confusion Matrix — Tuned Model (Akurasi: {acc_tuned*100:.2f}%)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### Perbandingan Performansi & Fitur Paling Berpengaruh (Feature Importance)

In [ ]:
# 1. Perbandingan Ringkas
metrics_base = {
    "Model": "Random Forest Base",
    "Akurasi": f"{accuracy_score(y_test, y_pred_base):.4f}",
    "F1 Macro": f"{f1_score(y_test, y_pred_base, average='macro'):.4f}",
    "F1 Weighted": f"{f1_score(y_test, y_pred_base, average='weighted'):.4f}",
    "Precision Macro": f"{precision_score(y_test, y_pred_base, average='macro'):.4f}",
    "Recall Macro": f"{recall_score(y_test, y_pred_base, average='macro'):.4f}",
}

metrics_tuned = {
    "Model": "Random Forest Tuned",
    "Akurasi": f"{accuracy_score(y_test, y_pred_tuned):.4f}",
    "F1 Macro": f"{f1_score(y_test, y_pred_tuned, average='macro'):.4f}",
    "F1 Weighted": f"{f1_score(y_test, y_pred_tuned, average='weighted'):.4f}",
    "Precision Macro": f"{precision_score(y_test, y_pred_tuned, average='macro'):.4f}",
    "Recall Macro": f"{recall_score(y_test, y_pred_tuned, average='macro'):.4f}",
}

df_compare = pd.DataFrame([metrics_base, metrics_tuned])
print("=" * 60)
print("PERBANDINGAN METRIK EVALUASI")
print("=" * 60)
print(df_compare.to_string(index=False))
print("=" * 60)

# 2. Plot Top-20 Feature Importance
importances = rf_tuned.feature_importances_
feat_names = tfidf.get_feature_names_out()
top_idx = np.argsort(importances)[::-1][:20]
df_imp = pd.DataFrame({
    "Fitur": feat_names[top_idx],
    "Importance": importances[top_idx],
})

plt.figure(figsize=(12, 7))
palette = sns.color_palette("flare", len(df_imp))
plt.barh(df_imp["Fitur"][::-1], df_imp["Importance"][::-1], color=palette)
plt.xlabel("Feature Importance Score", fontsize=11)
plt.title("Top-20 Fitur TF-IDF Paling Berpengaruh (Random Forest Tuned)", fontsize=12, fontweight="bold")

for i, val in enumerate(df_imp["Importance"][::-1]):
    plt.text(val + 0.0001, i, f"{val:.4f}", va="center", fontsize=8)
    
plt.tight_layout()
plt.show()


# 💾 SAVE MODEL

In [ ]:
MODEL_DIR = "saved_model"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "best_model.pkl")
tfidf_path = os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
le_path = os.path.join(MODEL_DIR, "label_encoder.pkl")
meta_path = os.path.join(MODEL_DIR, "metadata.json")

print("💾 Menyimpan model ke disk...")
joblib.dump(rf_tuned, model_path)
joblib.dump(tfidf, tfidf_path)
joblib.dump(le, le_path)

# Simpan metadata pendukung
metadata = {
    "created_at": datetime.now().isoformat(),
    "model_type": "RandomForestClassifier",
    "classes": le.classes_.tolist(),
    "label_mapping": {k: int(v) for k, v in label_mapping.items()},
    "best_params": {k: ("None" if v is None else v) for k, v in best_params.items()},
    "performance": {
        "base_model_accuracy": round(float(metrics_base["Akurasi"]), 4),
        "final_model_accuracy": round(float(metrics_tuned["Akurasi"]), 4),
        "f1_macro_final": round(float(metrics_tuned["F1 Macro"]), 4),
    }
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
    
print("✅ Semua berkas model berhasil disimpan di folder 'saved_model/'!")


# 🔮 INFERENCE

### Prediksi Sentimen Teks Baru Menggunakan Pendekatan Blending
Inference engine ini menggunakan pendekatan hybrid (blending): 50% probabilitas Random Forest (ML) digabung dengan 50% probabilitas Lexicon-based (InSet kustom). Model blending ini menghasilkan prediksi sentimen yang sangat handal dengan bias kelas minimal.

In [ ]:
class SentimentPredictor:
    def __init__(self, model_dir="saved_model"):
        best_model_path = os.path.join(model_dir, "best_model.pkl")
        tfidf_path = os.path.join(model_dir, "tfidf_vectorizer.pkl")
        le_path = os.path.join(model_dir, "label_encoder.pkl")
        
        self.model = joblib.load(best_model_path)
        self.tfidf = joblib.load(tfidf_path)
        self.label_encoder = joblib.load(le_path)
        self.preprocessor = TextPreprocessor()
        self.labeler = LexiconLabeler()

    def predict(self, text):
        if not isinstance(text, str) or not text.strip():
            return self._empty_result(text)
            
        # Proses preprocessing
        clean_text = self.preprocessor.filter_text(text)
        lower_text = clean_text.lower()
        normalized = self.preprocessor.normalize_words(lower_text)
        no_stop = self.preprocessor.remove_stopwords(normalized)
        stemmed = self.preprocessor.stem(no_stop)
        
        if not stemmed.strip():
            return self._empty_result(text)
            
        # 1. Probabilitas Leksikon (InSet)
        lex_score, _ = self.labeler.label_sentiment(stemmed)
        classes = self.label_encoder.classes_
        class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        
        lex_proba = np.zeros(len(classes))
        if lex_score > 0:
            lex_proba[class_to_idx.get("Positif", 0)] = 0.85 if lex_score < 5 else 0.95
            lex_proba[class_to_idx.get("Netral", 0)] = 0.10 if lex_score < 5 else 0.04
            lex_proba[class_to_idx.get("Negatif", 0)] = 0.05 if lex_score < 5 else 0.01
        elif lex_score < 0:
            lex_proba[class_to_idx.get("Negatif", 0)] = 0.85 if lex_score > -5 else 0.95
            lex_proba[class_to_idx.get("Netral", 0)] = 0.10 if lex_score > -5 else 0.04
            lex_proba[class_to_idx.get("Positif", 0)] = 0.05 if lex_score > -5 else 0.01
        else:
            lex_proba[class_to_idx.get("Netral", 0)] = 0.80
            lex_proba[class_to_idx.get("Positif", 0)] = 0.10
            lex_proba[class_to_idx.get("Negatif", 0)] = 0.10
            
        # 2. Probabilitas Model Machine Learning (TF-IDF + Random Forest)
        features = self.tfidf.transform([stemmed])
        model_proba = self.model.predict_proba(features)[0]
        
        # 3. Blending (Rata-rata 50/50)
        final_proba = 0.5 * model_proba + 0.5 * lex_proba
        pred_idx = np.argmax(final_proba)
        pred_label = classes[pred_idx]
        
        return {
            "text_original": text,
            "text_preprocessed": stemmed,
            "sentiment": str(pred_label),
            "confidence": round(float(final_proba.max()), 4),
            "probabilities": {
                str(cls): round(float(p), 4)
                for cls, p in zip(classes, final_proba)
            },
        }

    def _empty_result(self, text):
        classes = self.label_encoder.classes_
        return {
            "text_original": text,
            "text_preprocessed": "",
            "sentiment": "Netral",
            "confidence": 0.0,
            "probabilities": {str(cls): 0.0 for cls in classes},
        }


In [ ]:
print("🔮 Menginisialisasi SentimentPredictor...")
predictor = SentimentPredictor(model_dir="saved_model")

test_sentences = [
    # ── Contoh 1 : Sentimen Positif ──────────────────────────
    "Alhamdulillah akhirnya lolos LPDP! Bangga banget, "
    "programnya bener-bener membantu anak bangsa buat lanjut S2 ke luar negeri. "
    "Terima kasih LPDP, semangat buat yang masih berjuang!",

    # ── Contoh 2 : Sentimen Negatif ──────────────────────────
    "Proses seleksi LPDP sangat mengecewakan, dokumen adminnya ribet banget "
    "dan syaratnya sering berubah tanpa pemberitahuan yang jelas. "
    "Banyak pelamar berprestasi yang gagal hanya karena birokrasi amburadul.",

    # ── Contoh 3 : Sentimen Netral ───────────────────────────
    "LPDP membuka pendaftaran beasiswa reguler dalam negeri dan luar negeri. "
    "Kuota yang tersedia tahun ini sebanyak 4.000 awardee. "
    "Batas waktu pendaftaran adalah 30 Juni, persyaratan lengkap di web resmi LPDP.",
]

print("\n🎯 Hasil Pengujian Kalimat Contoh:")
print("=" * 75)
for i, sent in enumerate(test_sentences, 1):
    res = predictor.predict(sent)
    print(f"Test #{i}:")
    print(f"  📝 Kalimat : {res['text_original']}")
    print(f"  ⚙️ Bersih  : {res['text_preprocessed']}")
    print(f"  🎯 Kelas   : {res['sentiment']} (Confidence: {res['confidence']:.2%})")
    print(f"  📊 Proba   : {res['probabilities']}")
    print("=" * 75)
